# 04 - Explainability with SHAP

Requires `pip install shap` and a trained model (`python train.py --model hist_gb`). The scikit-learn-only analyses (permutation importance, partial dependence, error analysis) are already computed in `src/explainability/` and shown in the technical report; SHAP adds per-prediction attributions.

**Not executed in the environment that built the committed results** (`shap` was unavailable).

In [ ]:
import sys
sys.path.insert(0, "..")

import pickle
import numpy as np
import shap

from src.training_utils import load_training_frame, model_path

with open(model_path("hist_gb"), "rb") as f:
    pipe = pickle.load(f)

_, X_test, _, y_test = load_training_frame()
X = X_test.sample(500, random_state=42)

The saved pipeline is `features -> preprocess -> regressor`; SHAP's `TreeExplainer` needs the matrix the regressor actually sees.

In [ ]:
engineered = pipe.named_steps["features"].transform(X)
X_t = pipe.named_steps["preprocess"].transform(engineered)
names = [n.split("__", 1)[1] for n in pipe.named_steps["preprocess"].get_feature_names_out()]

explainer = shap.TreeExplainer(pipe.named_steps["regressor"])
shap_values = explainer.shap_values(X_t)

## Global importance

In [ ]:
shap.summary_plot(shap_values, X_t, feature_names=names, plot_type="bar")

## One listing

Expect `revpar_proxy` / occupancy / rate to dominate - see the ablation in the technical report for what happens without them.

In [ ]:
shap.force_plot(explainer.expected_value, shap_values[0], X_t[0], feature_names=names, matplotlib=True)